# NIRF Rankings 2017-2025: Merge, Check, Clean and Fill

This notebook merges the 9 yearly NIRF files, checks the data problems, fixes them, gives every institute one fixed ID, and fills all missing values.

**Steps:** 1. Load → 2. Check data → 3. Clean and make IDs → 4. Fill missing values → 5. Verify and save

| Column | Meaning |
|---|---|
| `year`, `category` | Ranking year and category (Overall, Engineering, ...) |
| `rank`, `score` | Rank and total score |
| `institute_name`, `city`, `state` | Institute details |
| `institute_id` | ID given by NIRF (changes between years) |
| `tlr`, `rpc`, `go`, `oi`, `pr` | Teaching, Research, Graduation Outcomes, Outreach, Perception scores |

## 1. Load

In [10]:
import re
import pandas as pd

group = ["year", "category"]                 # one ranking list = year + category
params = ["tlr", "rpc", "go", "oi", "pr"]    # parameter score columns

**Cell 1 - Imports and settings**
- `pandas` for the tables, `re` for cleaning text.
- `group` and `params` are used again and again below, so they are defined once here.

In [11]:
df = pd.concat([pd.read_csv(f"dataset/nirf_{y}.csv", encoding="utf-8-sig")
                for y in range(2017, 2026)], ignore_index=True)

# remove extra spaces
for col in ["institute_name", "city", "state"]:
    df[col] = df[col].str.strip()

print(df.shape)
df.head()

(6924, 13)


,year,category,rank,institute_name,city,state,score,institute_id,tlr,rpc,go,oi,pr
0,2017,Overall,1.0,Indian Institute of Science Bangalore,Bengaluru,Karnataka,83.28,IR17-I-2-18243,83.11,87.59,87.97,61.48,83.33
1,2017,Overall,2.0,Indian Institute of Technology Madras,Chennai,Tamil Nadu,73.97,IR17-I-1-1-77,69.49,72.60,84.02,76.75,68.70
2,2017,Overall,3.0,Indian Institute of Technology Bombay,Mumbai,Maharashtra,71.78,IR17-I-2-18633,64.68,78.14,70.07,65.80,83.44
3,2017,Overall,4.0,Indian Institute of Technology Kharagpur,Kharagpur,West Bengal,68.43,IR17-I-2-18630,55.07,70.46,85.11,72.85,64.72
4,2017,Overall,5.0,Indian Institute of Technology Delhi,New Delhi,Delhi,64.18,IR17-I-2-1-79,55.45,68.48,65.92,68.69,69.53


**Cell 2 - Load, merge and trim**
- Reads the 9 CSV files and stacks them into one table.
- `utf-8-sig` removes a hidden character at the start of each file that would otherwise rename the first column.
- Extra spaces are trimmed from the text columns.

## 2. Check the Data

In [12]:
df.info()
print()
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 6924 entries, 0 to 6923
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   year            6924 non-null   int64  
 1   category        6924 non-null   str    
 2   rank            6920 non-null   float64
 3   institute_name  6924 non-null   str    
 4   city            6924 non-null   str    
 5   state           6924 non-null   str    
 6   score           6923 non-null   float64
 7   institute_id    6924 non-null   str    
 8   tlr             6673 non-null   float64
 9   rpc             6673 non-null   float64
 10  go              6673 non-null   float64
 11  oi              6923 non-null   float64
 12  pr              6923 non-null   float64
dtypes: float64(7), int64(1), str(5)
memory usage: 1.2 MB

year                0
category            0
rank                4
institute_name      0
city                0
state               0
score               1
institute_id    

**Cell 3 - Types and missing values**
- `rank` is a float (`1.0`) because a few ranks are missing.
- Missing values exist in `rank`, `score`, `tlr`, `rpc`, `go`, `oi`, `pr`. They are explained in Cells 6-7 and filled in Section 4.

In [13]:
print("Full row duplicates:               ", df.duplicated().sum())
print("Same year + category + id:         ", df.duplicated(group + ["institute_id"]).sum())
print("Same year + category + name:       ", df.duplicated(group + ["institute_name"]).sum())
print("Same year + category + name + city:", df.duplicated(group + ["institute_name", "city"]).sum())

ranked = df.dropna(subset=["rank"])
print("Same year + category + rank:       ", ranked.duplicated(group + ["rank"]).sum())

Full row duplicates:                0
Same year + category + id:          0
Same year + category + name:        81
Same year + category + name + city: 0
Same year + category + rank:        168


**Cell 4 - Count every type of duplicate**

| Check | Meaning |
|---|---|
| Full row / same id | Should be 0: no true duplicates |
| Same name | Looks like duplicates (81 rows in the original data), but see Cell 5 |
| Same name + city | 0 means the same-name rows are different institutes in different cities |
| Same rank | Ties (168 rows in the original data), see Cell 6 |

In [14]:
same_name = df[df.duplicated(group + ["institute_name"], keep=False)]
same_name = same_name.sort_values(["institute_name", "year", "category"])
same_name[["year", "category", "rank", "institute_name", "city", "state"]]

,year,category,rank,institute_name,city,state
2288,2020,Pharmacy,58.0,Bharati Vidyapeeth's College of Pharmacy,Navi Mumbai,Maharashtra
2292,2020,Pharmacy,62.0,Bharati Vidyapeeth's College of Pharmacy,Kolhapur,Maharashtra
1568,2019,Pharmacy,49.0,Bharati Vidyapeeth`s College of Pharmacy,Navi Mumbai,Maharashtra
1573,2019,Pharmacy,54.0,Bharati Vidyapeeth`s College of Pharmacy,Kolhapur,Maharashtra
1697,2019,Medical,3.0,Christian Medical College,Vellore,Tamil Nadu
...,...,...,...,...,...,...
2052,2020,Engineering,97.0,University College of Engineering,Kakinada,Andhra Pradesh
5933,2024,Agriculture,11.0,University of Agricultural Sciences,Bengaluru,Karnataka
5946,2024,Agriculture,24.0,University of Agricultural Sciences,Dharwad,Karnataka
921,2018,College,22.0,Women`s Christian College,Chennai,Tamil Nadu


**Cell 5 - Look at the same-name cases**
- Each pair has the same name but a **different city**, e.g. *Holy Cross College* in Tiruchirappalli and in Kanyakumari.
- They are separate institutes, so they must **not** be dropped.
- Cell 11 keeps them apart by adding the city to the key.

In [15]:
ties = ranked[ranked.duplicated(group + ["rank"], keep=False)]
print("Rows involved in ties:", len(ties))

df[df["rank"].isna()][["year", "category", "rank", "institute_name", "score"]]

Rows involved in ties: 331


,year,category,rank,institute_name,score
721,2018,Engineering,NaN,Indian Institute of Engineering Science and Te...,53.24
727,2018,Engineering,NaN,Indian Institute of Technology Mandi,51.28
866,2018,Pharmacy,NaN,Banasthali Vidyapith,53.41
902,2018,College,NaN,Bishop Heber College,67.63


**Cell 6 - Rank ties and missing ranks**
- **Ties:** two institutes got the same score and share one rank. These are real, so they stay.
- NIRF does not skip the next rank after a tie, so a category can have more rows than its highest rank (e.g. 2020 Overall has 101 rows but ranks only go to 100).
- **Missing ranks:** 4 rows, all from 2018 (2 Engineering, 1 Pharmacy, 1 College), have no rank in the source file. They are filled in Section 4.

In [16]:
# rows with no score and no parameters at all
print(df[df["score"].isna()][["year", "category", "rank", "institute_name"]])

# which lists have missing tlr / rpc / go ?
print()
print(df[df["tlr"].isna()].groupby(group).size())

      year category  rank                                     institute_name
6761  2025   Dental  18.0  Faculty of Dental Sciences, Institute of Medic...

year  category
2021  Research    50
2022  Research    50
2023  Research    50
2024  Research    50
2025  Dental       1
      Research    50
dtype: int64


**Cell 7 - Missing scores**
- **2025 Dental, rank 18:** `score` and all 5 parameter columns are empty in the source.
- **Research category (2021-2025, 250 rows):** `tlr`, `rpc` and `go` are empty for every row. Only `oi` and `pr` are given, so these parameters are not measured for this category.

## 3. Clean and Make One ID per Institute

In [17]:
CITY_FIX = {
    "bangalore": "bengaluru", "mysore": "mysuru", "gurgaon": "gurugram",
    "calicut": "kozhikode", "trichur": "thrissur", "belgaum": "belagavi",
    "mangalore": "mangaluru", "mengaluru": "mangaluru",
    "tiruchirapalli": "tiruchirappalli", "allahabad": "prayagraj",
}
STATE_FIX = {"pondicherry": "puducherry"}


def clean_text(text):
    text = text.lower().replace("&", " and ")
    return re.sub(r"[^a-z0-9]+", " ", text).strip()     # removes . , ' ` ( ) -


def clean_city(city):
    city = clean_text(city)
    city = re.sub(r"\b(district|dist)\b", "", city)      # "Dist. Calicut" -> "calicut"
    city = re.sub(r"\s+", " ", city).strip()
    return CITY_FIX.get(city, city)


df["name_clean"] = df["institute_name"].map(clean_text)
df["city_clean"] = df["city"].map(clean_city)
df["state_clean"] = df["state"].map(clean_text).replace(STATE_FIX)

df[["institute_name", "name_clean", "city_clean", "state_clean"]].sample(5, random_state=1)

,institute_name,name_clean,city_clean,state_clean
6545,Sri Guru Gobind Singh College of Commerce,sri guru gobind singh college of commerce,delhi,delhi
5066,Banaras Hindu University,banaras hindu university,varanasi,uttar pradesh
4302,Thapar Institute of Engineering and Technology...,thapar institute of engineering and technology...,patiala,punjab
3144,"S.B. College, Changanacherry",s b college changanacherry,kottayam,kerala
5866,Government Dental College,government dental college,ahmedabad,gujarat


**Cell 8 - Cleaning functions**
- `clean_text` makes text comparable: lower case, no punctuation, `&` becomes `and`. So ``St. Joseph`s`` and `St. Joseph's` match.
- `clean_city` also removes "Dist." / "District" and uses `CITY_FIX` to join spelling variants (Bangalore / Bengaluru, Calicut / Kozhikode, ...).
- `STATE_FIX` joins `Pondicherry` and `Puducherry`. Lower-casing also joins `Jammu And Kashmir` and `Jammu and Kashmir`.
- The clean values go in **new columns**, so the original columns are untouched for now.

In [18]:
# "Indian Institute of Science, Bengaluru" -> "Indian Institute of Science"
# Only when the shorter name already exists in the same state.
known = set(zip(df["name_clean"], df["state_clean"]))

short_names = []
for name, city in zip(df["institute_name"], df["city"]):
    name = re.sub(r",\s*" + re.escape(city) + r"\s*$", "", name, flags=re.I)
    short_names.append(clean_text(name))

new_names = [short if (short, state) in known else old
             for short, old, state in zip(short_names, df["name_clean"], df["state_clean"])]

print("Names fixed:", (pd.Series(new_names) != df["name_clean"]).sum())
df["name_clean"] = new_names

Names fixed: 289


**Cell 9 - Names with the city added**
- Some years write `Indian Institute of Science, Bengaluru` and other years write `Indian Institute of Science`. They are the same institute.
- The city part is removed **only when** the shorter name already exists in the same state. This avoids damaging names like *University of Delhi*.

In [19]:
# key = name + state (state is safer than city because city spellings change a lot)
df["key"] = df["name_clean"] + " | " + df["state_clean"]

# if one key repeats in the same year + category, they are different institutes -> add city
clash = df.loc[df.duplicated(group + ["key"], keep=False), "key"].unique()
print("Names that clash:", len(clash))
for key in sorted(clash):
    print(" -", key)

df.loc[df["key"].isin(clash), "key"] += " | " + df["city_clean"]

# one new id per institute, same in every year
df["inst_id"] = df["key"].factorize()[0] + 1

Names that clash: 12
 - bharati vidyapeeth s college of pharmacy | maharashtra
 - government arts college | tamil nadu
 - government dental college | kerala
 - holy cross college | tamil nadu
 - indian institute of management | maharashtra
 - jaipuria institute of management | uttar pradesh
 - kasturba medical college | karnataka
 - kle college of pharmacy | karnataka
 - manipal college of dental sciences | karnataka
 - st joseph s college | kerala
 - university of agricultural sciences | karnataka
 - women s christian college | tamil nadu


**Cell 10 - Build one stable `inst_id` per institute**
- The old `institute_id` changes between years (2,144 different IDs for 1,318 names in the original data), so it cannot track an institute over time.
- New key = cleaned name + state. This joins the spelling variants from Cells 8-9.
- The clash list shows names that are really different institutes in the same state (e.g. *Government Dental College* in Kozhikode and Thiruvananthapuram). For only these, the city is added to the key.
- `factorize()` turns each key into a number: `inst_id`.

In [20]:
# use the latest year's spelling, so each institute has ONE name, city and state
df = df.sort_values("year", kind="stable")

for col in ["institute_name", "city", "state"]:
    df[col] = df.groupby("inst_id")[col].transform("last")

df[df["institute_name"].str.contains("Indian Institute of Science")][
    ["inst_id", "institute_name", "city", "state"]].drop_duplicates()

,inst_id,institute_name,city,state
0,1,Indian Institute of Science Bangalore,Bengaluru,Karnataka
28,29,Indian Institute of Science Education & Resear...,Pune,Maharashtra
43,44,Indian Institute of Science Education & Resear...,Mohanpur,West Bengal
51,52,Indian Institute of Science Education & Resear...,Mohali,Punjab
97,98,Indian Institute of Science Education & Resear...,Bhopal,Madhya Pradesh
500,357,Indian Institute of Science,Bengaluru,Karnataka
557,367,Indian Institute of Science Education & Resear...,Thiruvananthapuram,Kerala


**Cell 11 - One standard name, city and state per institute**
- Every row of an institute now uses its **latest year's** name, city and state.
- This removes leftover variants: `Bangalore` vs `Bengaluru`, backticks in names, `Pondicherry` vs `Puducherry`, and names with a city added.

## 4. Fill Missing Values

| Column | Problem | How it is filled |
|---|---|---|
| `rank` | missing in a few 2018 rows | worked out from `score` in the same year + category |
| `tlr`, `rpc`, `go` | empty for all Research rows (not measured) | `0` |
| `score` | missing in 1 row | taken from the rows ranked just above and below |
| other parameters | missing in 1 row | median of the same year + category |

The `filled` column marks every row that had a missing value.

In [21]:
value_cols = ["rank", "score"] + params
df["filled"] = df[value_cols].isnull().any(axis=1)      # remember which rows are fixed

# 1) rank: same score = same rank (NIRF does not skip a number after a tie)
df["rank"] = df["rank"].fillna(df.groupby(group)["score"].rank(method="dense", ascending=False))

# 2) Research has no tlr / rpc / go
df.loc[df["category"] == "Research", ["tlr", "rpc", "go"]] = 0

# 3) score: use the rows ranked just above and below
df = df.sort_values(group + ["rank"]).reset_index(drop=True)
df["score"] = df.groupby(group)["score"].transform(lambda s: s.interpolate(limit_direction="both"))

# 4) other parameters: median of the same year + category
for col in params:
    df[col] = df[col].fillna(df.groupby(group)[col].transform("median"))

df["rank"] = df["rank"].astype(int)

print("Rows filled:", df["filled"].sum())
print("Missing values left:", df[value_cols].isnull().sum().sum())

Rows filled: 255
Missing values left: 0


**Cell 12 - Fill every missing value**
- **rank:** same score = same rank, so a missing rank is worked out from the score.
- **Research:** `tlr`, `rpc`, `go` are not given for this category, so they are set to 0. Use the `filled` column if you want to leave these rows out of an average.
- **score:** a ranking list is sorted by score, so a missing score is taken from its neighbours.
- **Other parameters:** median of the same year + category, which is not affected by extreme values.
- The last line should print `0`.

## 5. Verify and Save

In [22]:
assert df.duplicated(group + ["inst_id"]).sum() == 0, "same institute twice in one list"
assert df.groupby("inst_id")["institute_name"].nunique().max() == 1, "one inst_id has 2 names"
assert df[value_cols].isnull().sum().sum() == 0, "missing values left"

print("Unique institutes:", df["inst_id"].nunique())
print("States:", df["state"].nunique())
print()
print("How many years each institute appears in:")
print(df.groupby("inst_id")["year"].nunique().value_counts().sort_index())

Unique institutes: 1124
States: 33

How many years each institute appears in:
year
1    311
2    171
3     96
4    110
5     90
6     78
7     48
8     89
9    131
Name: count, dtype: int64


**Cell 13 - Verify**
- The three `assert` lines stop the notebook with a message if something is wrong: a duplicate, an `inst_id` with two names, or a missing value.
- The last table shows how many institutes appear in 1 year, 2 years, ... 9 years.

In [23]:
final = df[["inst_id", "institute_name", "city", "state", "year", "category",
            "rank", "score"] + params + ["filled"]]
final = final.sort_values(["year", "category", "rank"]).reset_index(drop=True)

final.to_csv("nirf_merged.csv", index=False)
print("Saved:", final.shape)
final.head()

Saved: (6924, 14)


,inst_id,institute_name,city,state,year,category,rank,score,tlr,rpc,go,oi,pr,filled
0,257,Miranda House,Delhi,Delhi,2017,College,1,69.39,75.76,27.44,90.06,96.45,41.82,False
1,258,Loyola College,Chennai,Tamil Nadu,2017,College,2,68.68,66.92,100.00,48.42,56.45,80.45,False
2,259,Shri Ram College of Commerce,Delhi,Delhi,2017,College,3,67.18,77.63,0.20,97.51,93.80,53.98,False
3,260,Bishop Heber College,Tiruchirappalli,Tamil Nadu,2017,College,4,61.18,76.62,53.99,61.28,68.11,18.63,False
4,261,Atma Ram Sanatan Dharma College,New Delhi,Delhi,2017,College,5,60.68,74.22,30.54,79.60,82.51,0.21,False


**Cell 14 - Final table and save**
- Keeps 14 columns: institute info, `year`, `category`, `rank`, `score`, the 5 parameter scores and `filled`.
- The helper columns (`name_clean`, `key`, ...) and the old `institute_id` are dropped.
- Saved as `nirf_merged.csv`.

# Summary: All Issues, Reasons and Solutions

## A. Duplicate-looking cases

| # | Issue | Why it happens | Solution |
|---|---|---|---|
| 1 | **Same name appears twice** in one year and category | **Different institutes** with the same name in different cities, e.g. Holy Cross College | **Do not drop.** Add city to the key only for the names that clash (Cell 10) |
| 2 | **Same rank appears twice** | **Genuine ties.** Same score, so same rank | **Keep them** |
| 3 | **Full-row and same-ID duplicates** | None exist | Nothing to do |

## B. Inconsistent values

| # | Issue | Why it happens | Solution |
|---|---|---|---|
| 4 | **`institute_id` changes between years** | NIRF changed its ID format over the years | Make our own `inst_id` from name + state (Cell 10) |
| 5 | **Backtick vs apostrophe** `` St. Joseph`s `` vs `St. Joseph's` | Different typing in different years | `clean_text()` removes punctuation (Cell 8) |
| 6 | **City added to names** `Indian Institute of Science, Bengaluru` vs `Indian Institute of Science` | Naming style changed between years | Remove the city part when the shorter name already exists (Cell 9) |
| 7 | **City spelling varies** Bangalore / Bengaluru, Calicut / Kozhikode | Cities were renamed or written in different ways | `CITY_FIX` dictionary (Cell 8) |
| 8 | **"Dist." / "District" in cities** | Source formatting | `clean_city()` removes them (Cell 8) |
| 9 | **State spelling** `Pondicherry` vs `Puducherry`; `Jammu And Kashmir` vs `Jammu and Kashmir` | Official rename and capitalization | `STATE_FIX` and lower-casing (Cell 8) |
| 10 | **Same name, same state, different institutes** | Real separate institutes | City added to the key for these only (Cell 10) |
| 11 | **Extra spaces** in names, cities, states | Source formatting | `str.strip()` (Cell 2) |

## C. Missing values

| # | Issue | Why it happens | Solution |
|---|---|---|---|
| 12 | **Missing ranks** (2018) | Rank is empty in the source file | Worked out from `score` (Cell 12) |
| 13 | **1 row with no score and no parameters** (2025 Dental, rank 18) | Values are empty in the source file | `score` from neighbouring ranks, parameters from the median (Cell 12) |
| 14 | **`tlr`, `rpc`, `go` empty for all Research rows** (2021-2025) | Not given for this category | Filled with `0`; `filled` column marks them (Cell 12) |

## D. Formatting issues

| # | Issue | Why it happens | Solution |
|---|---|---|---|
| 15 | **Hidden BOM character** breaks the first column name | Files saved as UTF-8 with BOM | `encoding="utf-8-sig"` (Cell 2) |
| 16 | **Rank is a float** (`1.0`) | Different export format and the missing values | `astype(int)` after filling (Cell 12) |

## E. Things to know (not errors)

| # | Note | Explanation |
|---|---|---|
| 17 | **Some categories have 101 rows** (2020 Overall, 2022 Management, 2022 Pharmacy, 2025 Pharmacy) or fewer ranks than rows (2019 Overall, 2024 Law) | Ties do not skip the next rank, so one extra row appears |
| 18 | **Engineering has 200 rows in 2019 and 2020**, 100 in other years | NIRF listed more institutes in those two years |
| 19 | **Categories were added over time:** Law, Architecture, Medical (2018), Dental (2020), Research (2021), Agriculture (2023), State Public University (2024) | New NIRF categories. Do not compare a category across years before it existed |
| 20 | **The same institute appears in several categories in one year** | Normal, not a duplicate |

## F. Limits that remain

- Spelling mistakes in names (e.g. "Enterprenurship") and short forms like "(IIITDM)" are not fixed, so a few institutes may still be split in two.
- For the same-name-same-state institutes, city is part of the key, so a new city spelling not in `CITY_FIX` could split one of them.
- Filled values (rank, score, parameters) are estimates, not NIRF's published numbers. Use `filled` to find them.

## Order of the steps

1. **Load** (Cells 1-2) -> 2. **Check** (Cells 3-7) -> 3. **Clean and build `inst_id`** (Cells 8-11) -> 4. **Fill missing values** (Cell 12) -> 5. **Verify and save** (Cells 13-14)
